> # **Multi-Sensor SAVI Analysis and Time-Series Export**

> ## **Final Assessment**

> This notebook evaluates vegetation using the Soil-Adjusted Vegetation Index (SAVI) from Sentinel-2 and Landsat 8 imagery over the study area for the period 2016–2025.

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES AND INITIALISE GOOGLE EARTH ENGINE
# ============================================================

import ee
import geemap

# Initialise Google Earth Engine
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine initialised successfully.")

Google Earth Engine initialised successfully.


> ## **1. Study Area and Project Setup**

>The study area is the municipality of San Luis de Gaceno, located in the department of Boyacá, Colombia.

>The municipality was selected as the study area because it provides a defined administrative unit for evaluating vegetation conditions using multi-temporal satellite imagery. The study area geometry is obtained from the FAO GAUL Level 2 administrative boundaries dataset.

In [2]:
# ============================================================
# 2. DEFINE THE STUDY AREA
# ============================================================

# Load second-level administrative boundaries
gaul_level2 = ee.FeatureCollection("FAO/GAUL/2015/level2")

# Approximate centroid of San Luis de Gaceno, Boyacá, Colombia
study_point = ee.Geometry.Point([-73.1238, 4.8153])

# Select the municipality containing the study point
study_area = (
    gaul_level2
    .filter(ee.Filter.eq("ADM0_NAME", "Colombia"))
    .filterBounds(study_point)
)

# Geometry of the municipality
ee_muni_geom = study_area.geometry()

# Municipality centroid
centroid = ee_muni_geom.centroid()

print("Study area defined: San Luis de Gaceno, Boyacá, Colombia.")
print("Number of features found:", study_area.size().getInfo())

Study area defined: San Luis de Gaceno, Boyacá, Colombia.
Number of features found: 1


In [3]:
# ============================================================
# 3. CHECK THE STUDY AREA
# ============================================================

Map = geemap.Map(center=[4.8153, -73.1238], zoom=11)

Map.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area - San Luis de Gaceno"
)

Map

Map(center=[4.8153, -73.1238], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>### **Study Area Verification**

>The map above confirms the location and spatial extent of the selected municipality. The red boundary represents the study area used for all subsequent Sentinel-2 and Landsat 8 analyses.

In [4]:
# ============================================================
# 4. GENERAL PROJECT PARAMETERS
# ============================================================

# Study period
start_year = 2016
end_year = 2025

years = list(range(start_year, end_year + 1))

# SAVI soil adjustment factor
L = 0.5

# Required export CRS for Colombia
export_crs = "EPSG:9377"

print(f"Study period: {start_year}-{end_year}")
print(f"Years: {years}")
print(f"SAVI soil adjustment factor (L): {L}")
print(f"Export CRS: {export_crs}")

Study period: 2016-2025
Years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
SAVI soil adjustment factor (L): 0.5
Export CRS: EPSG:9377


>## **2. Sentinel-2 SAVI Analysis**

>Sentinel-2 Surface Reflectance imagery is used to calculate the Soil-Adjusted Vegetation Index (SAVI) over the study area.

>The analysis applies Sentinel-2 Scene Classification Layer (SCL) cloud masking, annual median compositing, and reflectance scaling before calculating SAVI.

> **Data availability note:** The Sentinel-2 Surface Reflectance Harmonized collection available in Google Earth Engine begins in 2017. Therefore, a valid Sentinel-2 SAVI composite cannot be generated for 2016 from this collection.

>### **2.1 Sentinel-2 Cloud Masking**

>Sentinel-2 imagery is filtered using the Scene Classification Layer (SCL) to remove cloud shadows, clouds, cirrus, and snow/ice pixels before calculating the annual SAVI composites.

In [5]:
# ============================================================
# 2.1 SENTINEL-2 CLOUD MASKING USING SCL
# ============================================================

def mask_s2_clouds_scl(image):
    """
    Mask clouds, cloud shadows, cirrus, and snow/ice
    using the Sentinel-2 Scene Classification Layer (SCL).
    """

    scl = image.select("SCL")

    # Mask unwanted SCL classes:
    # 3 = Cloud shadow
    # 8 = Medium probability cloud
    # 9 = High probability cloud
    # 10 = Cirrus
    # 11 = Snow/ice
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    # Select red and NIR bands and convert reflectance
    # from scaled integer values to surface reflectance.
    return (
        image
        .updateMask(mask)
        .select(["B4", "B8"])
        .divide(10000)
    )

print("Sentinel-2 SCL cloud masking function defined successfully.")

Sentinel-2 SCL cloud masking function defined successfully.


>### **2.2 Annual Sentinel-2 SAVI Composite**

>For each year, Sentinel-2 Surface Reflectance imagery is filtered to the study area and combined into an annual median composite. SAVI is then calculated from the red and near-infrared bands using a soil adjustment factor (L = 0.5).

In [6]:
# ============================================================
# 2.2 SENTINEL-2 ANNUAL SAVI COMPOSITE
# ============================================================

def compute_annual_savi_s2(year):
    """
    Generate an annual Sentinel-2 median composite
    and calculate SAVI for the study area.
    """

    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)

    # Sentinel-2 Surface Reflectance Harmonized collection
    annual_s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(mask_s2_clouds_scl)
    )

    # Annual median composite
    composite = annual_s2.median().clip(ee_muni_geom)

    # Calculate SAVI
    nir = composite.select("B8")
    red = composite.select("B4")

    savi = (
        nir.subtract(red)
        .divide(nir.add(red).add(L))
        .multiply(1 + L)
        .rename("SAVI")
    )

    return savi.set("year", year)


print("Sentinel-2 annual SAVI function defined successfully.")

Sentinel-2 annual SAVI function defined successfully.


>### **2.3 Sentinel-2 Data Availability**

>Before generating the annual SAVI time series, the availability of Sentinel-2 imagery is checked for each year from 2017 to 2025. This ensures that annual composites are generated only for years with available satellite observations.

In [9]:
# ============================================================
# 2.3 CHECK SENTINEL-2 DATA AVAILABILITY
# ============================================================

s2_years = list(range(2017, 2026))

print("Checking Sentinel-2 image availability:")

for year in s2_years:
    count = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(
            ee.Date.fromYMD(year, 1, 1),
            ee.Date.fromYMD(year + 1, 1, 1)
        )
        .size()
        .getInfo()
    )

    print(f"{year}: {count} images")

Checking Sentinel-2 image availability:
2017: 1 images
2018: 8 images
2019: 146 images
2020: 150 images
2021: 146 images
2022: 146 images
2023: 143 images
2024: 148 images
2025: 210 images


>### **2.4 Annual Sentinel-2 SAVI Time Series**

>Annual Sentinel-2 SAVI composites are generated for each year with available Surface Reflectance observations. For each year, cloud-masked imagery is combined using a pixel-wise median composite, and SAVI is calculated from the red and near-infrared bands.

In [10]:
# ============================================================
# 2.4 BUILD THE ANNUAL SENTINEL-2 SAVI TIME SERIES
# ============================================================

# Generate one annual SAVI image for each available year
s2_annual_savi = ee.ImageCollection.fromImages(
    [compute_annual_savi_s2(year) for year in s2_years]
)

print("Annual Sentinel-2 SAVI collection created.")
print("Number of annual images:", s2_annual_savi.size().getInfo())

Annual Sentinel-2 SAVI collection created.
Number of annual images: 9


>### **2.5 Sentinel-2 SAVI Visualization**

>The annual SAVI composite for 2025 is visualized to verify the spatial distribution of vegetation conditions within the study area.

In [11]:
# ============================================================
# 2.5 VISUALIZE SENTINEL-2 SAVI FOR 2025
# ============================================================

savi_2025_s2 = s2_annual_savi.filter(
    ee.Filter.eq("year", 2025)
).first()

savi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": ["brown", "yellow", "green"]
}

Map_savi_s2 = geemap.Map(
    center=[4.8153, -73.1238],
    zoom=11
)

Map_savi_s2.addLayer(
    savi_2025_s2,
    savi_vis,
    "Sentinel-2 SAVI 2025"
)

Map_savi_s2.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area"
)

Map_savi_s2

Map(center=[4.8153, -73.1238], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>## **3. Landsat 8 SAVI Analysis**

>Landsat 8 Surface Reflectance imagery is used to calculate the Soil-Adjusted Vegetation Index (SAVI) over the study area for the period 2016–2025.

>The analysis applies Landsat 8 QA_PIXEL cloud masking, Surface Reflectance scaling factors, annual median compositing, and SAVI calculation using the red and near-infrared bands.

>### **3.1 Landsat 8 Cloud Masking and Reflectance Scaling**

>Landsat 8 Collection 2 Level 2 imagery is filtered using the QA_PIXEL quality band to remove fill, dilated cloud, cirrus, cloud, cloud shadow, and snow pixels.

>The Surface Reflectance bands are then converted to physical reflectance values using the Landsat Collection 2 scaling factor and offset.

In [13]:
# ============================================================
# 3.1 LANDSAT 8 CLOUD MASKING AND REFLECTANCE SCALING
# ============================================================

def mask_l8_clouds(image):
    """
    Mask unwanted pixels using the Landsat 8 QA_PIXEL band
    and apply Collection 2 Surface Reflectance scaling.
    """

    qa = image.select("QA_PIXEL")

    # Mask unwanted QA_PIXEL classes:
    # Bit 0 = Fill
    # Bit 1 = Dilated cloud
    # Bit 2 = Cirrus
    # Bit 3 = Cloud
    # Bit 4 = Cloud shadow
    # Bit 5 = Snow
    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    # Select red and NIR bands and apply
    # Landsat Collection 2 Surface Reflectance scaling.
    return (
        image
        .updateMask(mask)
        .select(["SR_B4", "SR_B5"])
        .multiply(0.0000275)
        .add(-0.2)
    )

print("Landsat 8 QA_PIXEL cloud masking and scaling function defined successfully.")

Landsat 8 QA_PIXEL cloud masking and scaling function defined successfully.


>### **3.2 Annual Landsat 8 SAVI Composite**

>For each year from 2016 to 2025, Landsat 8 imagery is filtered to the study area and combined into an annual median composite. SAVI is then calculated using the red and near-infrared bands after cloud masking and reflectance scaling.

In [14]:
# ============================================================
# 3.2 LANDSAT 8 ANNUAL SAVI COMPOSITE
# ============================================================

def compute_annual_savi_l8(year):
    """
    Generate an annual Landsat 8 median composite
    and calculate SAVI for the study area.
    """

    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)

    # Landsat 8 Collection 2 Level 2 Surface Reflectance
    annual_l8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUD_COVER", 30))
        .map(mask_l8_clouds)
    )

    # Annual median composite
    composite = annual_l8.median().clip(ee_muni_geom)

    # Calculate SAVI
    nir = composite.select("SR_B5")
    red = composite.select("SR_B4")

    savi = (
        nir.subtract(red)
        .divide(nir.add(red).add(L))
        .multiply(1 + L)
        .rename("SAVI")
    )

    return savi.set("year", year)

print("Landsat 8 annual SAVI function defined successfully.")

Landsat 8 annual SAVI function defined successfully.


>### **3.3 Landsat 8 Data Availability**

>Landsat 8 image availability is checked for each year from 2016 to 2025. This confirms that annual observations are available throughout the study period before building the multi-temporal SAVI stack.

In [15]:
# ============================================================
# 3.3 CHECK LANDSAT 8 DATA AVAILABILITY
# ============================================================

l8_years = list(range(2016, 2026))

print("Checking Landsat 8 image availability:")

for year in l8_years:
    count = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(
            ee.Date.fromYMD(year, 1, 1),
            ee.Date.fromYMD(year + 1, 1, 1)
        )
        .size()
        .getInfo()
    )

    print(f"{year}: {count} images")

Checking Landsat 8 image availability:
2016: 19 images
2017: 19 images
2018: 17 images
2019: 18 images
2020: 17 images
2021: 19 images
2022: 16 images
2023: 18 images
2024: 19 images
2025: 13 images


>### **3.4 Annual Landsat 8 SAVI Time Series**

>Annual Landsat 8 SAVI composites are generated for each year from 2016 to 2025. For each year, cloud-masked imagery is combined using a pixel-wise median composite, and SAVI is calculated from the red and near-infrared bands.

In [16]:
# ============================================================
# 3.4 BUILD THE ANNUAL LANDSAT 8 SAVI TIME SERIES
# ============================================================

# Generate one annual SAVI image for each year from 2016 to 2025
l8_annual_savi = ee.ImageCollection.fromImages(
    [compute_annual_savi_l8(year) for year in l8_years]
)

print("Annual Landsat 8 SAVI collection created.")
print("Number of annual images:", l8_annual_savi.size().getInfo())

Annual Landsat 8 SAVI collection created.
Number of annual images: 10


>### **3.5 Landsat 8 SAVI Visualization**

>The 2025 annual Landsat 8 SAVI composite is visualized to verify the spatial distribution of vegetation conditions within the study area.

In [17]:
# ============================================================
# 3.5 VISUALIZE LANDSAT 8 SAVI FOR 2025
# ============================================================

savi_2025_l8 = l8_annual_savi.filter(
    ee.Filter.eq("year", 2025)
).first()

Map_savi_l8 = geemap.Map(
    center=[4.8153, -73.1238],
    zoom=11
)

Map_savi_l8.addLayer(
    savi_2025_l8,
    savi_vis,
    "Landsat 8 SAVI 2025"
)

Map_savi_l8.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area"
)

Map_savi_l8

Map(center=[4.8153, -73.1238], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>## **4. Multi-Temporal SAVI Stacks**

>The annual SAVI composites from Sentinel-2 and Landsat 8 are combined into multi-temporal image stacks covering 2016–2025.

>Each stack contains one annual SAVI layer per year, with bands named sequentially from SAVI_2016 to SAVI_2025.

> **Sentinel-2 2016 note:** The Sentinel-2 Surface Reflectance Harmonized collection begins in 2017. To provide the required 2016–2025 ten-year stack, Sentinel-2 Level-1C imagery is used only for 2016 with QA60 cloud masking. Surface Reflectance Harmonized imagery with SCL masking is used for 2017–2025.

>### **4.1 Sentinel-2 SAVI for 2016**

>Because Sentinel-2 Surface Reflectance data are not available for 2016, Sentinel-2 Level-1C imagery is used for this year only. The QA60 quality band is used to mask opaque clouds and cirrus before calculating the annual SAVI composite.

In [18]:
# ============================================================
# 4.1 SENTINEL-2 SAVI FOR 2016
# ============================================================

def mask_s2_2016(image):
    """
    Mask opaque clouds and cirrus using the Sentinel-2 QA60 band
    and convert reflectance values to surface reflectance scale.
    """

    qa = image.select("QA60")

    # Bit 10 = opaque clouds
    # Bit 11 = cirrus
    cloud_mask = (
        qa.bitwiseAnd(1 << 10).eq(0)
        .And(qa.bitwiseAnd(1 << 11).eq(0))
    )

    # Select red and NIR bands and convert
    # scaled reflectance values.
    return (
        image
        .updateMask(cloud_mask)
        .select(["B4", "B8"])
        .divide(10000)
    )


# Sentinel-2 Level-1C imagery for 2016
s2_2016 = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterBounds(ee_muni_geom)
    .filterDate("2016-01-01", "2017-01-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
    .map(mask_s2_2016)
)

# Annual median composite
s2_2016_composite = s2_2016.median().clip(ee_muni_geom)

# Calculate SAVI
savi_2016_s2 = (
    s2_2016_composite.select("B8")
    .subtract(s2_2016_composite.select("B4"))
    .divide(
        s2_2016_composite.select("B8")
        .add(s2_2016_composite.select("B4"))
        .add(L)
    )
    .multiply(1 + L)
    .rename("SAVI")
    .set("year", 2016)
)

print("Sentinel-2 2016 SAVI composite created successfully.")

Sentinel-2 2016 SAVI composite created successfully.


>### **4.2 Multi-Temporal SAVI Stacks**

>The annual SAVI composites are combined into one ten-band image for each sensor. Each band represents one year from 2016 to 2025 and is named sequentially from SAVI_2016 to SAVI_2025.

In [19]:
# ============================================================
# 4.2 BUILD THE MULTI-TEMPORAL SAVI STACKS
# ============================================================

# ------------------------------------------------------------
# Sentinel-2 stack: 2016–2025
# ------------------------------------------------------------

# Combine the 2016 composite with the 2017–2025 composites
s2_savi_images = [savi_2016_s2]

for year in range(2017, 2026):
    image = s2_annual_savi.filter(
        ee.Filter.eq("year", year)
    ).first()

    s2_savi_images.append(image)

# Create the ten-band Sentinel-2 stack
s2_savi_stack = (
    ee.ImageCollection.fromImages(s2_savi_images)
    .toBands()
    .rename([f"SAVI_{year}" for year in range(2016, 2026)])
)


# ------------------------------------------------------------
# Landsat 8 stack: 2016–2025
# ------------------------------------------------------------

# Create the ten-band Landsat 8 stack
l8_savi_stack = (
    l8_annual_savi
    .sort("year")
    .toBands()
    .rename([f"SAVI_{year}" for year in range(2016, 2026)])
)


# ------------------------------------------------------------
# Verify the stacks
# ------------------------------------------------------------

print("Sentinel-2 SAVI stack created.")
print("Sentinel-2 bands:", s2_savi_stack.bandNames().getInfo())

print("\nLandsat 8 SAVI stack created.")
print("Landsat 8 bands:", l8_savi_stack.bandNames().getInfo())

print("\nSentinel-2 number of bands:", s2_savi_stack.bandNames().size().getInfo())
print("Landsat 8 number of bands:", l8_savi_stack.bandNames().size().getInfo())

Sentinel-2 SAVI stack created.
Sentinel-2 bands: ['SAVI_2016', 'SAVI_2017', 'SAVI_2018', 'SAVI_2019', 'SAVI_2020', 'SAVI_2021', 'SAVI_2022', 'SAVI_2023', 'SAVI_2024', 'SAVI_2025']

Landsat 8 SAVI stack created.
Landsat 8 bands: ['SAVI_2016', 'SAVI_2017', 'SAVI_2018', 'SAVI_2019', 'SAVI_2020', 'SAVI_2021', 'SAVI_2022', 'SAVI_2023', 'SAVI_2024', 'SAVI_2025']

Sentinel-2 number of bands: 10
Landsat 8 number of bands: 10


>## **5. GeoTIFF Export**

>The ten-band Sentinel-2 and Landsat 8 SAVI stacks are exported as projected GeoTIFF rasters to Google Drive.

>Both outputs use EPSG:9377 as the required projection for the final raster products.

>### **5.1 Export Sentinel-2 SAVI Stack**

>The Sentinel-2 ten-band SAVI stack is exported to Google Drive as a GeoTIFF raster using EPSG:9377.

In [20]:
# ============================================================
# 5.1 EXPORT SENTINEL-2 SAVI STACK
# ============================================================

s2_export_task = ee.batch.Export.image.toDrive(
    image=s2_savi_stack,
    description="Sentinel2_SAVI_Stack_2016_2025",
    folder="GEE_SAVI_Final",
    fileNamePrefix="Sentinel2_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=10,
    crs=export_crs,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

s2_export_task.start()

print("Sentinel-2 SAVI GeoTIFF export task started.")

Sentinel-2 SAVI GeoTIFF export task started.


>### **5.2 Export Landsat 8 SAVI Stack**

>The Landsat 8 ten-band SAVI stack is exported to Google Drive as a GeoTIFF raster using EPSG:9377.

In [21]:
# ============================================================
# 5.2 EXPORT LANDSAT 8 SAVI STACK
# ============================================================

l8_export_task = ee.batch.Export.image.toDrive(
    image=l8_savi_stack,
    description="Landsat8_SAVI_Stack_2016_2025",
    folder="GEE_SAVI_Final",
    fileNamePrefix="Landsat8_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=30,
    crs=export_crs,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

l8_export_task.start()

print("Landsat 8 SAVI GeoTIFF export task started.")

Landsat 8 SAVI GeoTIFF export task started.


>### **5.3 Export Task Verification**

>The Earth Engine export tasks are checked to confirm that both SAVI GeoTIFF exports have been successfully submitted for processing.

In [22]:
# ============================================================
# 5.3 VERIFY EXPORT TASKS
# ============================================================

tasks = ee.batch.Task.list()

print("===== EXPORT TASK STATUS =====")

for task in tasks:
    description = task.config.get("description", "")
    
    if description in [
        "Sentinel2_SAVI_Stack_2016_2025",
        "Landsat8_SAVI_Stack_2016_2025"
    ]:
        print(description)
        print("Status:", task.status()["state"])
        print()

===== EXPORT TASK STATUS =====
Landsat8_SAVI_Stack_2016_2025
Status: FAILED

Sentinel2_SAVI_Stack_2016_2025
Status: FAILED



### **5.4 Export Correction and CRS Verification**

>The initial Sentinel-2 and Landsat 8 export tasks failed because Earth Engine could not parse the CRS definition when EPSG:9377 was provided directly during the export.

>The ten-band SAVI stacks were successfully created and were not affected by this export issue. The problem was limited to the projection parameter used during GeoTIFF export.

>The export tasks were therefore resubmitted using the corresponding PROJ.4 definition of EPSG:9377. The original failed tasks are retained in the notebook as part of the processing record, while the corrected tasks are used for the final GeoTIFF outputs.

In [23]:
# ============================================================
# CHECK EXPORT ERROR MESSAGES
# ============================================================

tasks = ee.batch.Task.list()

for task in tasks:
    description = task.config.get("description", "")

    if description in [
        "Sentinel2_SAVI_Stack_2016_2025",
        "Landsat8_SAVI_Stack_2016_2025"
    ]:
        status = task.status()

        print("========================================")
        print("Task:", description)
        print("Status:", status.get("state"))
        print("Error:", status.get("error_message", "No error message"))
        print("========================================")

Task: Landsat8_SAVI_Stack_2016_2025
Status: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: Sentinel2_SAVI_Stack_2016_2025
Status: FAILED
Error: Projection: The CRS of a map projection could not be parsed.


In [25]:
# ============================================================
# 5.4 CORRECTED EXPORT: SENTINEL-2 SAVI STACK
# ============================================================

# PROJ.4 definition corresponding to EPSG:9377
epsg_9377_proj4 = (
    "+proj=tmerc "
    "+lat_0=4 "
    "+lon_0=-73 "
    "+k=0.9992 "
    "+x_0=5000000 "
    "+y_0=2000000 "
    "+ellps=GRS80 "
    "+towgs84=0,0,0,0,0,0,0 "
    "+units=m "
    "+no_defs "
    "+type=crs"
)

s2_export_task_corrected = ee.batch.Export.image.toDrive(
    image=s2_savi_stack,
    description="Sentinel2_SAVI_Stack_2016_2025_Corrected",
    folder="GEE_SAVI_Final",
    fileNamePrefix="Sentinel2_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=10,
    crs=epsg_9377_proj4,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

s2_export_task_corrected.start()

print("Corrected Sentinel-2 SAVI GeoTIFF export task started.")
print("Export CRS: EPSG:9377")

Corrected Sentinel-2 SAVI GeoTIFF export task started.
Export CRS: EPSG:9377


>### **5.5 Corrected Landsat 8 GeoTIFF Export**

>The Landsat 8 ten-band SAVI stack is resubmitted for export using the corresponding PROJ.4 definition of EPSG:9377 to ensure that the required projection is correctly interpreted by Earth Engine.

In [26]:
# ============================================================
# 5.5 CORRECTED EXPORT: LANDSAT 8 SAVI STACK
# ============================================================

l8_export_task_corrected = ee.batch.Export.image.toDrive(
    image=l8_savi_stack,
    description="Landsat8_SAVI_Stack_2016_2025_Corrected",
    folder="GEE_SAVI_Final",
    fileNamePrefix="Landsat8_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=30,
    crs=epsg_9377_proj4,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

l8_export_task_corrected.start()

print("Corrected Landsat 8 SAVI GeoTIFF export task started.")
print("Export CRS: EPSG:9377")

Corrected Landsat 8 SAVI GeoTIFF export task started.
Export CRS: EPSG:9377


In [27]:
# ============================================================
# 5.6 VERIFY CORRECTED EXPORT TASKS
# ============================================================

tasks = ee.batch.Task.list()

print("===== CORRECTED EXPORT TASK STATUS =====")

for task in tasks:
    description = task.config.get("description", "")

    if description in [
        "Sentinel2_SAVI_Stack_2016_2025_Corrected",
        "Landsat8_SAVI_Stack_2016_2025_Corrected"
    ]:
        status = task.status()

        print("Task:", description)
        print("Status:", status.get("state"))
        print("Error:", status.get("error_message", "No error"))
        print()

===== CORRECTED EXPORT TASK STATUS =====
Task: Landsat8_SAVI_Stack_2016_2025_Corrected
Status: FAILED
Error: Projection: The CRS of a map projection could not be parsed.

Task: Sentinel2_SAVI_Stack_2016_2025_Corrected
Status: FAILED
Error: Projection: The CRS of a map projection could not be parsed.

Task: Sentinel2_SAVI_Stack_2016_2025_Corrected
Status: FAILED
Error: Projection: The CRS of a map projection could not be parsed.



>### **5.7 CRS Definition Using WKT**

>The corrected export tasks continued to return a CRS parsing error when the EPSG:9377 projection was provided directly or through a PROJ.4 definition.

>To resolve the projection parsing issue, EPSG:9377 is defined explicitly using its WKT representation. This preserves the required MAGNA-SIRGAS 2018 / Origen-Nacional coordinate reference system.

In [28]:
# ============================================================
# 5.7 DEFINE EPSG:9377 USING WKT
# ============================================================

epsg_9377_wkt = """
PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",
    GEOGCS["MAGNA-SIRGAS 2018",
        DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018",
            SPHEROID["GRS 1980",6378137,298.257222101,
                AUTHORITY["EPSG","7019"]],
            TOWGS84[0,0,0,0,0,0,0]
        ],
        PRIMEM["Greenwich",0,
            AUTHORITY["EPSG","8901"]
        ],
        UNIT["degree",0.0174532925199433,
            AUTHORITY["EPSG","9122"]
        ],
        AUTHORITY["EPSG","20046"]
    ],
    PROJECTION["Transverse_Mercator"],
    PARAMETER["latitude_of_origin",4],
    PARAMETER["central_meridian",-73],
    PARAMETER["scale_factor",0.9992],
    PARAMETER["false_easting",5000000],
    PARAMETER["false_northing",2000000],
    UNIT["metre",1,
        AUTHORITY["EPSG","9001"]
    ],
    AUTHORITY["EPSG","9377"]
]
"""

print("EPSG:9377 WKT definition created successfully.")

EPSG:9377 WKT definition created successfully.


>### **5.8 CRS Projection Validation**

>The WKT definition of EPSG:9377 is tested using the Earth Engine projection object before submitting new export tasks.

>The successful validation confirms that Earth Engine can parse the MAGNA-SIRGAS 2018 / Origen-Nacional projection correctly.

In [29]:
# ============================================================
# 5.8 TEST EPSG:9377 WKT PROJECTION
# ============================================================

try:
    test_proj = ee.Projection(epsg_9377_wkt)

    print("Projection created successfully.")
    print(test_proj.getInfo())

except Exception as e:
    print("Projection error:")
    print(e)

Projection created successfully.
{'type': 'Projection', 'wkt': 'PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional", \n  GEOGCS["MAGNA-SIRGAS 2018", \n    DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018", \n      SPHEROID["GRS 1980", 6378137.0, 298.257222101, AUTHORITY["EPSG","7019"]], \n      TOWGS84[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], \n    PRIMEM["Greenwich", 0.0, AUTHORITY["EPSG","8901"]], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH], \n    AUTHORITY["EPSG","20046"]], \n  PROJECTION["Transverse_Mercator"], \n  PARAMETER["central_meridian", -73.0], \n  PARAMETER["latitude_of_origin", 4.0], \n  PARAMETER["scale_factor", 0.9992], \n  PARAMETER["false_easting", 5000000.0], \n  PARAMETER["false_northing", 2000000.0], \n  UNIT["m", 1.0], \n  AXIS["x", EAST], \n  AXIS["y", NORTH], \n  AUTHORITY["EPSG","9377"]]', 'transform': [1, 0, 0, 0, 1, 0]}


>### **5.9 Sentinel-2 SAVI Stack Export Using Validated CRS**

>The Sentinel-2 ten-band SAVI stack is exported to Google Drive as a GeoTIFF using the validated WKT representation of EPSG:9377.

>The export uses the native Sentinel-2 spatial resolution of 10 metres.

In [30]:
# ============================================================
# 5.9 EXPORT SENTINEL-2 SAVI STACK USING WKT
# ============================================================

s2_export_task_wkt = ee.batch.Export.image.toDrive(
    image=s2_savi_stack,
    description="Sentinel2_SAVI_Stack_2016_2025_WKT",
    folder="GEE_SAVI_Final",
    fileNamePrefix="Sentinel2_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=10,
    crs=epsg_9377_wkt,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

s2_export_task_wkt.start()

print("Sentinel-2 SAVI GeoTIFF export task started successfully.")
print("Projection: EPSG:9377")
print("Scale: 10 m")

Sentinel-2 SAVI GeoTIFF export task started successfully.
Projection: EPSG:9377
Scale: 10 m


In [31]:
# ============================================================
# 5.10 VERIFY SENTINEL-2 EXPORT TASK
# ============================================================

status = s2_export_task_wkt.status()

print("===== SENTINEL-2 EXPORT STATUS =====")
print("Task:", status.get("description"))
print("Status:", status.get("state"))
print("Error:", status.get("error_message", "No error"))
print("====================================")

===== SENTINEL-2 EXPORT STATUS =====
Task: Sentinel2_SAVI_Stack_2016_2025_WKT
Status: RUNNING
Error: No error


>### **5.11 Landsat 8 SAVI Stack Export Using Validated CRS**

>The Landsat 8 ten-band SAVI stack is exported to Google Drive as a GeoTIFF using the validated WKT representation of EPSG:9377.

>The export uses the native Landsat 8 spatial resolution of 30 metres.

In [33]:
# ============================================================
# 5.11 EXPORT LANDSAT 8 SAVI STACK USING WKT
# ============================================================

l8_export_task_wkt = ee.batch.Export.image.toDrive(
    image=l8_savi_stack,
    description="Landsat8_SAVI_Stack_2016_2025_WKT",
    folder="GEE_SAVI_Final",
    fileNamePrefix="Landsat8_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=30,
    crs=epsg_9377_wkt,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

l8_export_task_wkt.start()

print("Landsat 8 SAVI GeoTIFF export task started successfully.")
print("Projection: EPSG:9377")
print("Scale: 30 m")

Landsat 8 SAVI GeoTIFF export task started successfully.
Projection: EPSG:9377
Scale: 30 m


In [34]:
# ============================================================
# 5.12 VERIFY LANDSAT 8 EXPORT TASK
# ============================================================

status = l8_export_task_wkt.status()

print("===== LANDSAT 8 EXPORT STATUS =====")
print("Task:", status.get("description"))
print("Status:", status.get("state"))
print("Error:", status.get("error_message", "No error"))
print("===================================")

===== LANDSAT 8 EXPORT STATUS =====
Task: Landsat8_SAVI_Stack_2016_2025_WKT
Status: READY
Error: No error


>### **5.13 Final Export Task Verification**

>The Sentinel-2 and Landsat 8 SAVI GeoTIFF export tasks are verified to confirm that both outputs were successfully submitted to Google Earth Engine using the required EPSG:9377 projection.

In [35]:
# ============================================================
# 5.14 FINAL EXPORT TASK VERIFICATION
# ============================================================

tasks = ee.batch.Task.list()

print("===== FINAL EXPORT TASK VERIFICATION =====")

for task in tasks:
    description = task.config.get("description", "")

    if description in [
        "Sentinel2_SAVI_Stack_2016_2025_WKT",
        "Landsat8_SAVI_Stack_2016_2025_WKT"
    ]:
        status = task.status()

        print("Task:", description)
        print("Status:", status.get("state"))
        print("Error:", status.get("error_message", "No error"))
        print("----------------------------------------")

print("Both final export tasks use EPSG:9377.")

===== FINAL EXPORT TASK VERIFICATION =====
Task: Landsat8_SAVI_Stack_2016_2025_WKT
Status: COMPLETED
Error: No error
----------------------------------------
Task: Landsat8_SAVI_Stack_2016_2025_WKT
Status: COMPLETED
Error: No error
----------------------------------------
Task: Sentinel2_SAVI_Stack_2016_2025_WKT
Status: COMPLETED
Error: No error
----------------------------------------
Both final export tasks use EPSG:9377.
